# 📚 Módulo 04 - Modelos de Series Temporales

## 🎯 Objetivos

1. ✅ Entender componentes de series temporales
2. ✅ Aplicar modelos ARIMA, Prophet, LSTM
3. ✅ Evaluar forecasts
4. ✅ Manejar estacionalidad y tendencias

---

## 1️⃣ Componentes de Series Temporales

### Descomposición

```
Y(t) = Tendencia(t) + Estacionalidad(t) + Ruido(t)
```

* **Tendencia**: Dirección a largo plazo
* **Estacionalidad**: Patrones que se repiten periódicamente
* **Ruido**: Variación aleatoria

```python
from statsmodels.tsa.seasonal import seasonal_decompose

decomposition = seasonal_decompose(ts, model='additive', period=12)
decomposition.plot()
```

---

## 2️⃣ ARIMA (AutoRegressive Integrated Moving Average)

### Componentes

* **AR(p)**: AutoRegressive - usa valores pasados
* **I(d)**: Integrated - diferenciación para estacionariedad
* **MA(q)**: Moving Average - usa errores pasados

### Implementación

```python
from statsmodels.tsa.arima.model import ARIMA

# Ajustar modelo ARIMA(1,1,1)
model = ARIMA(ts, order=(1, 1, 1))
model_fit = model.fit()

# Forecast
forecast = model_fit.forecast(steps=12)
print(forecast)
```

### Selección de parámetros (p, d, q)

```python
import pmdarima as pm

# Auto ARIMA
auto_model = pm.auto_arima(
    ts,
    start_p=0, start_q=0,
    max_p=5, max_q=5,
    seasonal=True, m=12,
    stepwise=True,
    suppress_warnings=True
)

print(auto_model.summary())
```

---

## 3️⃣ Prophet (Facebook)

**Ventajas:**
* Maneja outliers automáticamente
* Captura múltiples estacionalidades
* Incluye holidays
* Robusto ante datos faltantes

```python
from prophet import Prophet
import pandas as pd

# Preparar datos (requiere 'ds' y 'y')
df_prophet = pd.DataFrame({
    'ds': ts.index,
    'y': ts.values
})

# Ajustar modelo
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False
)
model.fit(df_prophet)

# Forecast
future = model.make_future_dataframe(periods=365)
forecast = model.predict(future)

# Visualizar
model.plot(forecast)
model.plot_components(forecast)
```

### Holidays y eventos

```python
holidays = pd.DataFrame({
    'holiday': 'navidad',
    'ds': pd.to_datetime(['2024-12-25', '2025-12-25']),
    'lower_window': -2,
    'upper_window': 2
})

model = Prophet(holidays=holidays)
```

---

## 4️⃣ LSTM para Series Temporales

```python
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Preparar datos (ventana deslizante)
def create_sequences(data, window_size=10):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)

window_size = 10
X, y = create_sequences(ts.values, window_size)

# Reshape para LSTM: (samples, timesteps, features)
X = X.reshape(X.shape[0], X.shape[1], 1)

# Definir modelo
model = keras.Sequential([
    layers.LSTM(50, return_sequences=True, input_shape=(window_size, 1)),
    layers.LSTM(50),
    layers.Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.fit(X, y, epochs=50, batch_size=32, verbose=0)

# Forecast
last_window = ts.values[-window_size:].reshape(1, window_size, 1)
prediction = model.predict(last_window)
```

---

## 5️⃣ Evaluación de Forecasts

### Métricas

```python
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")
```

### Train/Test Split Temporal

```python
# NO usar shuffle=True!
train_size = int(len(ts) * 0.8)
train = ts[:train_size]
test = ts[train_size:]
```

---

## 6️⃣ Cuándo Usar Cada Método

| Método | Ventajas | Desventajas | Uso |
|---------|----------|-------------|-----|
| ARIMA | Interpretable, rápido | Requiere estacionariedad | Series simples |
| Prophet | Robusto, fácil | Menos preciso | Business forecasting |
| LSTM | Captura patrones complejos | Requiere muchos datos | Series complejas |

---

## ✅ Mejores Prácticas

✅ **Explorar primero**: Plot, descomposición, ACF/PACF
✅ **Validación temporal**: Train/test/validation splits secuenciales
✅ **Ensemble**: Combinar múltiples modelos
✅ **Actualizar regularmente**: Reentrenar con datos nuevos
✅ **Monitorear drift**: Performance degrada con el tiempo

---

**Universidad del Aconcagua 🇦🇷**